## Text input

In [1]:
from langchain.agents import create_agent
from azure_openai_llm import create_azure_llm
from langchain.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv()

llm = create_azure_llm()

agent = create_agent(llm,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [2]:
question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Welcome to **Lunaris Prime**, the shining capital of The Moon!

Perched within a massive, transparent dome on the edge of the Sea of Tranquility, Lunaris Prime is a marvel of human ingenuity and lunar adaptation. The city’s architecture blends sleek, reflective silver habitats with bioengineered lunar flora, creating a harmonious balance between technology and nature.

Key features include:

- **The Celestial Spire**: A towering communications and research tower that stretches skyward, facilitating real-time data exchange between Earth, lunar outposts, and deep-space missions.

- **Helios Plaza**: The bustling central hub where residents gather, featuring markets, entertainment, and panoramic views of Earth suspended in the black lunar sky.

- **Regolith Gardens**: Vast indoor botanical gardens that recycle waste and produce oxygen, sustaining the city’s atmosphere and food supply.

- **The Lunar Nexus Transit System**: A network of magnetic levitation tunnels connecting Lunaris Prime 

## Image input

In [3]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)
print(uploader.value)

FileUpload(value=(), accept='.png', description='Upload')

()


In [4]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [5]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this diagram"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

This diagram illustrates a simplified flow of a system involving a "Model" and "Tools" interacting to process a "Request" and produce a "Result." Here's a breakdown of the components and their interactions:

1. Request (Blue Box): This is the initial input or query that the system receives, likely from a user or an external source (represented by the robot icon).

2. Model (Purple Box): The core component that receives the request. The model processes this request and interacts with tools to carry out actions.

3. Tools (Red Box): These are external functionalities or utilities that the model can use to perform specific actions. The model sends "Action" commands to the tools.

4. Interaction between Model and Tools: The model sends an action to the tools, and the tools provide observations back to the model. This suggests a feedback loop where the model can adjust based on the observations made from the tools' actions.

5. Result (Gray Box): After processing, the model produces a final

In [6]:
response['messages'][-1].response_metadata

{'token_usage': {'completion_tokens': 263,
  'prompt_tokens': 1300,
  'total_tokens': 1563,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-4.1-mini-2025-04-14',
 'system_fingerprint': 'fp_b6f445fc1c',
 'id': 'chatcmpl-DQoclgyUceyHAw4Lt2AwhqIrvnRXB',
 'prompt_filter_results': [{'prompt_index': 0,
   'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'},
    'jailbreak': {'filtered': False, 'detected': False},
    'self_harm': {'filtered': False, 'severity': 'safe'},
    'sexual': {'filtered': False, 'severity': 'safe'},
    'violence': {'filtered': False, 'severity': 'safe'}}}],
 'finish_reason': 'stop',
 'logprobs': None,
 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'},
  'protected_material_code': {'filtered': False, 'd

In [ ]:
# Check what deployments you have available
import os
from dotenv import load_dotenv

load_dotenv()

print("🔍 Current Azure AI Foundry Configuration:")
print("=" * 50)
print(f"Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Chat Deployment: {os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}")
print(f"Audio Deployment: {os.getenv('AZURE_OPENAI_AUDIO_DEPLOYMENT')}")
print(f"API Version: {os.getenv('AZURE_OPENAI_API_VERSION')}")
print("=" * 50)

# Test if we can create connections to both deployments
print("\n📋 Testing Model Availability:")
print("-" * 30)

try:
    from azure_openai_llm import create_azure_llm
    
    print("✅ Chat model connection:")
    chat_llm = create_azure_llm(type="chat")
    print(f"   Model type: {type(chat_llm).__name__}")
    
    print("\n✅ Audio model connection:")  
    audio_llm = create_azure_llm(type="audio")
    print(f"   Model type: {type(audio_llm).__name__}")
    
except Exception as e:
    print(f"❌ Error: {e}")

In [10]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm # tqdm provides a visual progress bar during the recording loop.

# Steps:
# 2) Record audio from the microphone using sounddevice (sd).
# 3) Display a short progress bar during recording.
# 4) Wait for recording completion to ensure data is finalized.
# 5) Convert WAV bytes to Base64 encoded string.

# Recording settings
duration = 3  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 30/30 [00:03<00:00,  9.88it/s]


Done.


In [11]:
from langchain.agents import create_agent
from azure_openai_llm import create_azure_llm
from langchain.messages import HumanMessage

agent = create_agent(create_azure_llm(type="audio", api="openai"))

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Follow the instructions in the audio and reply"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Using audio deployment: lums-gpt-audio-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview
The result of adding 2 and 4 is 6.


In [18]:
response['messages'][-1].response_metadata

{'token_usage': {'completion_tokens': 12,
  'prompt_tokens': 49,
  'total_tokens': 61,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0,
   'text_tokens': 12},
  'prompt_tokens_details': {'audio_tokens': 30,
   'cached_tokens': 0,
   'image_tokens': 0,
   'text_tokens': 19}},
 'model_provider': 'openai',
 'model_name': 'gpt-audio-mini-2025-12-15',
 'system_fingerprint': 'fp_94129ed17c',
 'id': 'chatcmpl-DOH1jNUYNEjNV9EvZ4Tqoi7t5cmzj',
 'prompt_filter_results': [{'prompt_index': 0,
   'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'},
    'jailbreak': {'filtered': False, 'detected': False},
    'self_harm': {'filtered': False, 'severity': 'safe'},
    'sexual': {'filtered': False, 'severity': 'safe'},
    'violence': {'filtered': False, 'severity': 'safe'}}}],
 'finish_reason': 'stop',
 'logprobs': None,
 'content_filter_results': {'hate': {'filtered': False, 'seve